# Train the screen-element detector in Google Colab

**Md Meraj Ansari · Symbiosis Institute of Technology, Hyderabad**

This notebook fine-tunes YOLO11n on reviewed screenshots for `Speaker_Video`, `Presentation_Slide`, and `Screen_Share`.

1. Collect and annotate 100–200 screenshots locally, across independent sessions.
2. Run `python -m semantic_dataset prepare`, then `python -m semantic_dataset export`.
3. Open this notebook in Colab and select **Runtime → Change runtime type → GPU**.
4. Upload your generated `training_bundle.zip` when prompted below.

The bundle contains project validation/training Python source. Use the archive generated by this project. This notebook imports those helpers after extraction. Synthetic tooling fixtures are not training data.

References: [YOLO annotation format](https://docs.ultralytics.com/datasets/detect), [YOLO11](https://docs.ultralytics.com/models/yolo11), [training](https://docs.ultralytics.com/modes/train), [validation](https://docs.ultralytics.com/modes/val).


## 1. Install the pinned training environment

The local desktop recorder does not need Torch or Ultralytics. This cell installs the GPU-training dependencies in Colab and checks that CUDA is available.


In [ ]:
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics==8.4.175", "PyYAML>=6,<7", "Pillow>=10,<13"])
import torch
import ultralytics
print("Ultralytics:", ultralytics.__version__, "Torch:", torch.__version__)
assert torch.cuda.is_available(), "Choose a GPU runtime, then restart/run these cells."
print("GPU:", torch.cuda.get_device_name(0))


## 2. Upload and extract the prepared bundle

Only this step uploads your screenshots to Colab. Review and anonymize meeting content before building the dataset. The extraction checks reject path traversal, symlinks, duplicate archive paths, and unexpectedly large archives.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import os
import shutil
import tempfile
import zipfile
uploaded = files.upload()
assert len(uploaded) == 1, "Upload one training_bundle.zip."
archive_name = next(iter(uploaded))
assert archive_name.lower().endswith(".zip"), "Expected a ZIP bundle."
WORK = Path(tempfile.mkdtemp(prefix="screen-elements-", dir="/content"))
helpers = {"semantic_dataset/__init__.py", "semantic_dataset/core.py", "training/__init__.py",
           "training/train_yolo.py", "requirements-yolo.txt"}
with zipfile.ZipFile(archive_name) as archive:
    entries = archive.infolist()
    assert sum(item.file_size for item in entries) <= 2 * 1024**3, "Bundle exceeds 2 GiB."
    seen = set()
    for item in entries:
        name = item.filename
        relative = PurePosixPath(name)
        assert not relative.is_absolute() and ".." not in relative.parts, "Unsafe archive path."
        assert "\\" not in name and ":" not in name, "Unsafe archive filename."
        assert name not in seen, "Duplicate archive entry."
        seen.add(name)
        assert ((item.external_attr >> 16) & 0o170000) != 0o120000, "Symlinks are not supported."
        if item.is_dir():
            continue
        is_data = relative.parts[0] == "screen_elements" and relative.suffix in {".png", ".txt", ".yaml", ".json"}
        assert is_data or name in helpers, "Unexpected file in bundle: " + name
        target = WORK.joinpath(*relative.parts)
        assert target.resolve().is_relative_to(WORK.resolve()), "Archive path escaped the workspace."
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(item) as source, target.open("xb") as destination:
            shutil.copyfileobj(source, destination)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
DATASET = WORK / "screen_elements"
print("Extracted to:", WORK)


## 3. Validate the real dataset before training

The project requires 100 reviewed images by default, all three classes in every split, and isolation of whole sessions. Missing labels are rejected rather than assumed to be negative. Exact duplicate images and modified image hashes are rejected. Grouping prevents adjacent frames from the same session appearing in validation or test.


In [ ]:
import json
from semantic_dataset.core import validate_prepared
DATASET_REPORT, errors = validate_prepared(DATASET, min_images=100)
print(json.dumps(DATASET_REPORT, indent=2))
assert not errors, "Dataset failed validation:\n" + "\n".join(errors[:20])


## 4. Train the small baseline model

Start with YOLO11n at 640 px. At this dataset size, accuracy is exploratory: expand the data when rare layouts or tiny camera tiles are missed. Text-bearing screenshots should not be mirrored, rotated, or combined with mosaic, so the recipe disables those augmentations. Seed/settings and dataset/checkpoint hashes are saved with the results.

Adjust batch size if GPU memory is insufficient. Tune using validation data; keep test data reserved until settings are final. Deterministic settings improve repeatability but do not guarantee identical results across different GPU/software versions.


In [ ]:
from training.train_yolo import train
TRAIN_CONFIG = dict(model="yolo11n.pt", epochs=100, imgsz=640, batch=16, device="0", seed=42)
TRAIN_REPORT = train(DATASET, WORK / "runs", **TRAIN_CONFIG, evaluate_test=False)
print(json.dumps(TRAIN_REPORT, indent=2))
BEST = Path(TRAIN_REPORT["checkpoint"])
assert BEST.is_file(), "Training did not produce best.pt."


## 5. Inspect validation predictions

Check slide/camera/app boundaries, false positives, unreadable small tiles, and layouts the model misses. Region detection alone does not establish a frame-retention policy or storage/power savings.


In [ ]:
from ultralytics import YOLO
from PIL import Image
from IPython.display import display
import cv2
best_model = YOLO(str(BEST))
validation_images = sorted((DATASET / "images" / "val").glob("*.png"))[:4]
for image_path in validation_images:
    result = best_model.predict(str(image_path), conf=0.35, imgsz=TRAIN_CONFIG["imgsz"], verbose=False)[0]
    print(image_path.name, result.speed)
    display(Image.fromarray(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)))


## 6. Optional final held-out test

Set the switch to `True` once after the model and settings have been selected using validation. Repeated tuning against this test turns it into another validation set.


In [ ]:
EVALUATE_HELD_OUT_TEST = False
if EVALUATE_HELD_OUT_TEST:
    import yaml
    from semantic_dataset import CLASSES
    from semantic_dataset.core import write_json
    runtime_yaml = WORK / "held_out_data.yaml"
    runtime_yaml.write_text(yaml.safe_dump({"path": str(DATASET), "train": "images/train",
        "val": "images/val", "test": "images/test", "names": dict(enumerate(CLASSES))}, sort_keys=False))
    test_metrics = best_model.val(data=str(runtime_yaml), split="test", imgsz=TRAIN_CONFIG["imgsz"],
                                  batch=TRAIN_CONFIG["batch"], device="0", project=str(WORK / "runs"), name="held_out_test")
    TRAIN_REPORT["held_out_test"] = dict(test_metrics.results_dict)
    write_json(BEST.parent.parent / "training_report.json", TRAIN_REPORT)
    print(json.dumps(TRAIN_REPORT["held_out_test"], indent=2))
else:
    print("Held-out test remains reserved; use validation predictions and metrics first.")


## 7. Download the checkpoint and evidence

Download before disconnecting the Colab runtime. The archive includes `best.pt`, the training report, the split manifest, and available learning-curve/validation plots. Put the checkpoint in the local project's `models/` folder for the future semantic-policy evaluation milestone.


In [ ]:
export_path = WORK / "screen_elements_model.zip"
with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(BEST, "weights/best.pt")
    archive.write(BEST.parent.parent / "training_report.json", "training_report.json")
    archive.write(DATASET / "split_manifest.json", "split_manifest.json")
    archive.write(DATASET / "dataset_report.json", "dataset_report.json")
    result_directory = BEST.parent.parent
    for pattern in ("results.csv", "*.png", "args.yaml"):
        for path in sorted(result_directory.glob(pattern)):
            archive.write(path, "training/" + path.name)
print("Saved:", export_path)
files.download(str(export_path))
